In [1]:
import pandas as pd
import re
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

STOPWORDS = set(stopwords.words('english'))

## Load and prepare data

In [2]:
df = pd.read_csv('../data/LabeledText.csv', encoding='latin-1')
df = df[['Caption', 'LABEL']].copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

df['clean'] = df['Caption'].apply(clean_text)

# Test set is locked in here — identical to all other notebooks
X_trainval, X_test, y_trainval, y_test = train_test_split(
    df['clean'], df['LABEL'],
    test_size=0.2,
    random_state=42,
    stratify=df['LABEL']
)

# Val set carved from training portion only
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.2,
    random_state=42,
    stratify=y_trainval
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

LookupError: 
**********************************************************************
  Resource [93mpunkt[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt/english.pickle[0m

  Searched in:
    - 'C:\\Users\\jonah/nltk_data'
    - 'C:\\Users\\jonah\\anaconda3\\nltk_data'
    - 'C:\\Users\\jonah\\anaconda3\\share\\nltk_data'
    - 'C:\\Users\\jonah\\anaconda3\\lib\\nltk_data'
    - 'C:\\Users\\jonah\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
    - ''
**********************************************************************


## Tune C on validation set

In [ ]:
C_values = [0.01, 0.1, 1, 10, 100]
val_results = []

for C in C_values:
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', LogisticRegression(C=C, max_iter=1000))
    ])
    pipeline.fit(X_train, y_train)
    val_acc = accuracy_score(y_val, pipeline.predict(X_val))
    val_results.append({'C': C, 'Val Accuracy': val_acc})
    print(f'C={C:<6}  Val Accuracy: {val_acc:.3f}')

best_C = max(val_results, key=lambda x: x['Val Accuracy'])['C']
print(f'\nBest C: {best_C}')

## Retrain on full train+val, evaluate on test

In [ ]:
best_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(C=best_C, max_iter=1000))
])
best_pipeline.fit(X_trainval, y_trainval)
y_pred = best_pipeline.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred))

joblib.dump(best_pipeline, '../models/text/logistic_regression.pkl')